# 03a — Silver Layer: Dimension Tables (dim_coin, dim_date, dim_date_hour)

In [0]:
# ══════════════════════════════════════════════════════════════════════════
# SILVER LAYER — DIMENSION TABLES
# New Data Model: Star Schema
#
#   dim_coin          — Master coin attributes (one row per coin)
#   dim_date          — Date dimension (calendar + trading day attributes)
#   dim_date_hour     — Hour-level dimension (for OHLC grain)
#
# Gold layer inputs remain IDENTICAL — silver_market_metrics &
# silver_ohlc_metrics are re-created as views on top of these dims +
# fact tables, so gold layer notebooks need zero changes.
# ══════════════════════════════════════════════════════════════════════════

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType,
    BooleanType, DateType, TimestampType
)
from datetime import datetime, timezone, date, timedelta

DB_NAME = 'crypto_db'

# ── Dimension target tables ───────────────────────────────────────────────
DIM_COIN      = f'{DB_NAME}.dim_coin'
DIM_DATE      = f'{DB_NAME}.dim_date'
DIM_DATE_HOUR = f'{DB_NAME}.dim_date_hour'

# ── Source bronze tables (read only) ─────────────────────────────────────
BRONZE_MARKET = f'{DB_NAME}.bronze_market_data'
BRONZE_OHLC   = f'{DB_NAME}.bronze_ohlc_data'

def _now_utc():
    return datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S')

def _today_utc():
    return datetime.now(timezone.utc).strftime('%Y-%m-%d')

print('Config loaded.')
for t in [DIM_COIN, DIM_DATE, DIM_DATE_HOUR]:
    print(f'  Target: {t}')

In [0]:
# ══════════════════════════════════════════════════════════════════════════
# DIM 1 — dim_coin
# Grain : 1 row per coin_id (natural key = id from CoinGecko)
# Contains: static + slowly-changing coin metadata
# SCD Type: Type-1 (overwrite) — we always want the latest rank, name etc.
# ══════════════════════════════════════════════════════════════════════════

def build_dim_coin():
    """
    Coin master dimension.
    Pulls the most recent snapshot per coin from bronze_market_data.
    All purely static / slowly-changing attributes live here so that
    fact tables carry only the surrogate / natural key.
    """

    df = spark.table(BRONZE_MARKET)

    # Standardise column names
    df = df.toDF(*[c.lower().strip().replace(' ', '_') for c in df.columns])

    # Drop noisy / pipeline cols we do not want in master
    DROP_COLS = ['image', 'roi', 'is_fallback', 'archive_run_id',
                 'price_change_percentage_24h_in_currency']
    df = df.drop(*[c for c in DROP_COLS if c in df.columns])

    # Cast
    NUMERIC_COLS = [
        'current_price', 'market_cap', 'total_volume',
        'circulating_supply', 'total_supply', 'max_supply',
        'ath', 'ath_change_percentage', 'atl', 'atl_change_percentage',
        'fully_diluted_valuation'
    ]
    for c in NUMERIC_COLS:
        if c in df.columns:
            df = df.withColumn(c, F.col(c).cast(DoubleType()))

    if 'market_cap_rank' in df.columns:
        df = df.withColumn('market_cap_rank', F.col('market_cap_rank').cast(IntegerType()))

    if 'ingestion_timestamp' in df.columns:
        df = df.withColumn('ingestion_timestamp', F.col('ingestion_timestamp').cast(TimestampType()))

    # Trim strings
    for col_name, dtype in df.dtypes:
        if dtype == 'string':
            df = df.withColumn(col_name, F.trim(F.col(col_name)))

    # Keep only the latest row per coin (most recent ingestion_date)
    from pyspark.sql.window import Window
    w_latest = Window.partitionBy('id').orderBy(F.col('ingestion_date').desc())
    df = (df
        .withColumn('_rn', F.row_number().over(w_latest))
        .filter(F.col('_rn') == 1)
        .drop('_rn')
    )

    # ── Market segment (derived, stored in dim for convenience) ───────────
    df = df.withColumn(
        'market_segment',
        F.when(F.col('market_cap_rank') <= 10,  'Top_10')
         .when(F.col('market_cap_rank') <= 50,  'Top_50')
         .when(F.col('market_cap_rank') <= 200, 'Top_200')
         .otherwise('Long_Tail')
    )

    # ── Supply utilisation ratio ──────────────────────────────────────────
    df = df.withColumn(
        'supply_utilisation_pct',
        F.when(
            F.col('max_supply').isNotNull() & (F.col('max_supply') > 0),
            F.round(F.col('circulating_supply') / F.col('max_supply') * 100, 2)
        )
    )

    # ── Scarcity flag (circulating > 90% of max) ─────────────────────────
    df = df.withColumn(
        'is_scarce',
        F.when(F.col('supply_utilisation_pct') >= 90, True).otherwise(False)
    )

    # ── ATH distance bucket ───────────────────────────────────────────────
    df = df.withColumn(
        'ath_distance_bucket',
        F.when(F.col('ath_change_percentage') >= -5,   'AT_ATH')
         .when(F.col('ath_change_percentage') >= -20,  'NEAR_ATH')
         .when(F.col('ath_change_percentage') >= -50,  'MID_RANGE')
         .otherwise('DEEP_CORRECTION')
    )

    # ── DWH audit cols ────────────────────────────────────────────────────
    df = (df
        .withColumn('dim_coin_updated_at', F.lit(_now_utc()).cast(TimestampType()))
        .withColumn('dim_coin_source',     F.lit('bronze_market_data'))
    )

    return df.select(
        # Natural key
        'id',
        # Identity
        'symbol', 'name',
        # Market rank / segment (SCD-1: overwrite on every run)
        'market_cap_rank', 'market_segment',
        # Supply economics
        'circulating_supply', 'total_supply', 'max_supply',
        'supply_utilisation_pct', 'is_scarce',
        # ATH / ATL benchmarks
        'ath', 'ath_change_percentage', 'ath_date',
        'atl', 'atl_change_percentage', 'atl_date',
        'ath_distance_bucket',
        # Valuation reference
        'fully_diluted_valuation',
        # Audit
        'dim_coin_updated_at', 'dim_coin_source',
    )


df_dim_coin = build_dim_coin()
coin_cnt = df_dim_coin.count()
print(f'dim_coin rows: {coin_cnt}')

# UPSERT — merge on id so SCD-1 is maintained cleanly
df_dim_coin.createOrReplaceTempView('_dim_coin_src')

if not spark.catalog.tableExists(DIM_COIN):
    (df_dim_coin.write.format('delta')
        .mode('overwrite')
        .option('overwriteSchema', 'true')
        .saveAsTable(DIM_COIN))
    print(f'Created {DIM_COIN}')
else:
    tgt_cols = set(spark.table(DIM_COIN).columns)
    src_cols = set(df_dim_coin.columns)
    common   = list(src_cols & tgt_cols)
    set_c    = ', '.join([f'tgt.{c} = src.{c}' for c in common])
    ins_c    = ', '.join(common)
    ins_v    = ', '.join([f'src.{c}' for c in common])
    spark.sql(f"""
        MERGE INTO {DIM_COIN} tgt
        USING _dim_coin_src src
        ON tgt.id = src.id
        WHEN MATCHED     THEN UPDATE SET {set_c}
        WHEN NOT MATCHED THEN INSERT ({ins_c}) VALUES ({ins_v})
    """)
    print(f'Merged {DIM_COIN}')

spark.sql(f'OPTIMIZE {DIM_COIN} ZORDER BY (market_cap_rank)')
print(f'✅ {DIM_COIN} ready — {spark.table(DIM_COIN).count()} rows')
spark.table(DIM_COIN).select(
    'id','symbol','market_cap_rank','market_segment',
    'supply_utilisation_pct','ath_distance_bucket'
).orderBy('market_cap_rank').display(10, truncate=False)

In [0]:
# ══════════════════════════════════════════════════════════════════════════
# DIM 2 — dim_date
# Grain  : 1 row per calendar date
# Range  : from earliest bronze ingestion_date → today + 90d buffer
# Purpose: join key for all daily-grain fact tables
# ══════════════════════════════════════════════════════════════════════════

def build_dim_date(start_date: date, end_date: date):
    """
    Pure calendar dimension — no dependency on coin data.
    Generates one row per day between start_date and end_date inclusive.
    """

    # Build list of dates
    delta  = (end_date - start_date).days + 1
    dates  = [start_date + timedelta(days=i) for i in range(delta)]

    rows = []
    for d in dates:
        iso_wd    = d.isoweekday()                         # Mon=1 … Sun=7
        week_num  = int(d.strftime('%W'))                  # week of year
        quarter   = (d.month - 1) // 3 + 1

        rows.append({
            # Surrogate / natural key
            'date_id':             int(d.strftime('%Y%m%d')),   # 20240101
            'date_actual':         d.isoformat(),               # 2024-01-01
            # Calendar breakdown
            'year':                d.year,
            'quarter':             quarter,
            'quarter_label':       f'Q{quarter}',
            'month':               d.month,
            'month_name':          d.strftime('%B'),
            'month_abbr':          d.strftime('%b'),
            'week_of_year':        week_num,
            'day_of_year':         d.timetuple().tm_yday,
            'day_of_month':        d.day,
            'day_of_week':         iso_wd,                      # 1=Mon 7=Sun
            'day_name':            d.strftime('%A'),
            'day_abbr':            d.strftime('%a'),
            # Trading-day flags
            'is_weekend':          iso_wd >= 6,
            'is_weekday':          iso_wd <= 5,
            # Crypto trades 24/7 — always true; kept for BI filter compatibility
            'is_trading_day':      True,
            # Period labels for grouping in BI tools
            'year_month':          d.strftime('%Y-%m'),         # 2024-01
            'year_quarter':        f'{d.year}-Q{quarter}',      # 2024-Q1
            'year_week':           d.strftime('%Y-W%W'),        # 2024-W01
            # Relative flags (computed once at dim build — refresh daily)
            'is_today':            d == date.today(),
            'is_past':             d < date.today(),
            'is_future':           d > date.today(),
        })

    schema = StructType([
        StructField('date_id',       IntegerType(),  False),
        StructField('date_actual',   StringType(),   False),
        StructField('year',          IntegerType(),  False),
        StructField('quarter',       IntegerType(),  False),
        StructField('quarter_label', StringType(),   False),
        StructField('month',         IntegerType(),  False),
        StructField('month_name',    StringType(),   False),
        StructField('month_abbr',    StringType(),   False),
        StructField('week_of_year',  IntegerType(),  False),
        StructField('day_of_year',   IntegerType(),  False),
        StructField('day_of_month',  IntegerType(),  False),
        StructField('day_of_week',   IntegerType(),  False),
        StructField('day_name',      StringType(),   False),
        StructField('day_abbr',      StringType(),   False),
        StructField('is_weekend',    BooleanType(),  False),
        StructField('is_weekday',    BooleanType(),  False),
        StructField('is_trading_day',BooleanType(),  False),
        StructField('year_month',    StringType(),   False),
        StructField('year_quarter',  StringType(),   False),
        StructField('year_week',     StringType(),   False),
        StructField('is_today',      BooleanType(),  False),
        StructField('is_past',       BooleanType(),  False),
        StructField('is_future',     BooleanType(),  False),
    ])

    return spark.createDataFrame(rows, schema=schema)


# Determine date range from bronze data
bm = spark.table(BRONZE_MARKET)
bm = bm.toDF(*[c.lower().strip() for c in bm.columns])

earliest_str = bm.agg(F.min('ingestion_date')).collect()[0][0]
earliest_dt  = datetime.strptime(str(earliest_str), '%Y-%m-%d').date()
end_dt       = date.today() + timedelta(days=90)   # 90-day future buffer

print(f'Date range: {earliest_dt} → {end_dt}  ({(end_dt - earliest_dt).days + 1} days)')

df_dim_date = build_dim_date(earliest_dt, end_dt)

(df_dim_date.write.format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable(DIM_DATE))

spark.sql(f'OPTIMIZE {DIM_DATE} ZORDER BY (date_id)')
print(f'✅ {DIM_DATE} ready — {spark.table(DIM_DATE).count()} rows')
spark.table(DIM_DATE).orderBy('date_actual').display(5, truncate=False)

In [0]:
# ══════════════════════════════════════════════════════════════════════════
# DIM 3 — dim_date_hour
# Grain  : 1 row per (date, hour) — 24 rows per day
# Purpose: join key for OHLC fact table (4h candles resolved to hour start)
# ══════════════════════════════════════════════════════════════════════════

def build_dim_date_hour(start_date: date, end_date: date):
    """
    Hour-level date dimension.
    Extends dim_date with intraday hour attributes (0–23).
    Only hours that align to OHLC candle boundaries (0,4,8,12,16,20)
    carry is_candle_open=True, but all 24 are generated for flexibility.
    """

    CANDLE_HOURS = {0, 4, 8, 12, 16, 20}   # 4-hour candle boundaries

    delta = (end_date - start_date).days + 1
    rows  = []

    for day_offset in range(delta):
        d = start_date + timedelta(days=day_offset)
        for h in range(24):
            quarter = (d.month - 1) // 3 + 1
            rows.append({
                'date_hour_id':   int(d.strftime('%Y%m%d')) * 100 + h,  # 2024010108
                'date_id':        int(d.strftime('%Y%m%d')),
                'date_actual':    d.isoformat(),
                'hour':           h,
                'hour_label':     f'{h:02d}:00',
                'candle_block':   (h // 4) * 4,            # 0,4,8,12,16,20
                'is_candle_open': h in CANDLE_HOURS,
                # Session labels (UTC-based, rough crypto convention)
                'trading_session':
                    'ASIA'    if 0  <= h < 8  else
                    'LONDON'  if 8  <= h < 13 else
                    'NY'      if 13 <= h < 21 else
                    'OFF',
                # Calendar carry-forward for convenience
                'year':           d.year,
                'month':          d.month,
                'day_of_week':    d.isoweekday(),
                'is_weekend':     d.isoweekday() >= 6,
            })

    schema = StructType([
        StructField('date_hour_id',    IntegerType(), False),
        StructField('date_id',         IntegerType(), False),
        StructField('date_actual',     StringType(),  False),
        StructField('hour',            IntegerType(), False),
        StructField('hour_label',      StringType(),  False),
        StructField('candle_block',    IntegerType(), False),
        StructField('is_candle_open',  BooleanType(), False),
        StructField('trading_session', StringType(),  False),
        StructField('year',            IntegerType(), False),
        StructField('month',           IntegerType(), False),
        StructField('day_of_week',     IntegerType(), False),
        StructField('is_weekend',      BooleanType(), False),
    ])

    return spark.createDataFrame(rows, schema=schema)


df_dim_date_hour = build_dim_date_hour(earliest_dt, end_dt)

(df_dim_date_hour.write.format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable(DIM_DATE_HOUR))

spark.sql(f'OPTIMIZE {DIM_DATE_HOUR} ZORDER BY (date_id, hour)')
print(f'✅ {DIM_DATE_HOUR} ready — {spark.table(DIM_DATE_HOUR).count()} rows')
spark.table(DIM_DATE_HOUR).filter('date_actual = "2024-01-01"').show(24, truncate=False)

In [0]:
# ── Final verification ────────────────────────────────────────────────────

print('\n====== DIMENSION LAYER VERIFICATION ======')

print('\n--- dim_coin: top 5 by rank ---')
spark.table(DIM_COIN).orderBy('market_cap_rank').select(
    'id','symbol','name','market_cap_rank','market_segment',
    'supply_utilisation_pct','ath_distance_bucket','is_scarce'
).display(5, truncate=False)

print('\n--- dim_date: sample rows ---')
spark.table(DIM_DATE).filter('is_today = true').display(1, truncate=False)

print('\n--- dim_date_hour: candle open hours only ---')
spark.table(DIM_DATE_HOUR).filter('is_candle_open = true').limit(6).display(truncate=False)

print('\n--- Row counts ---')
for t in [DIM_COIN, DIM_DATE, DIM_DATE_HOUR]:
    print(f'  {t}: {spark.table(t).count()}')

print('\n✅ Dimension layer complete.')

In [0]:
print("de")